In [1]:
import numpy as np

In [2]:
#тут я просто копипащу код из первого задания для линейного штрафа, я тупой, я думал руками сделать это значит самому накодить ыыыы

In [4]:
#needleman_wunsch algorythm
x = 'ATGCAGCAGCAGCCA'
y = 'ATATAT'
def needleman_wunsch(X, Y, match, mismatch, gap):
    #тут я делаю остов нашей матрицы
    X = list(X)
    Y = list(Y)
    matrix = np.zeros((len(X) + 2, len(Y) + 2), dtype = object)
    counter = gap
    for i in range(len(X)):
        matrix[i + 2, 1] += counter
        counter += gap
        matrix[i + 2, 0] = X[i]
    counter = gap
    for i in range(len(Y)):
        matrix[1, i + 2] += counter
        counter += gap
        matrix[0, i + 2] = Y[i]
    #тут начинаем заполнять
    for i in range(len(X)):
        for j in range(len(Y)):
            #match
            if matrix[i + 2, 0] == matrix[0, j + 2]:
                matrix[i + 2, j + 2] = max([matrix[i + 1,j + 2] + gap, matrix[i + 2, j + 1] + gap, matrix[i + 1, j + 1] + match])
                #print(matrix[i + 2, 0], matrix[0, j +2])
            #mismatch
            else: #matrix[i + 2, 0] != matrix[0, j + 2]:
                matrix[i + 2, j + 2] = max([matrix[i + 1,j + 2] + gap, matrix[i + 2, j + 1] + gap, matrix[i + 1, j + 1] + mismatch])
    #print(matrix)
    #traceback time
    return matrix

In [5]:
needleman_wunsch(x, y, 3, -3, -4)

array([[0, 0, 'A', 'T', 'A', 'T', 'A', 'T'],
       [0, 0, -4, -8, -12, -16, -20, -24],
       ['A', -4, 3, -1, -5, -9, -13, -17],
       ['T', -8, -1, 6, 2, -2, -6, -10],
       ['G', -12, -5, 2, 3, -1, -5, -9],
       ['C', -16, -9, -2, -1, 0, -4, -8],
       ['A', -20, -13, -6, 1, -3, 3, -1],
       ['G', -24, -17, -10, -3, -2, -1, 0],
       ['C', -28, -21, -14, -7, -6, -5, -4],
       ['A', -32, -25, -18, -11, -10, -3, -7],
       ['G', -36, -29, -22, -15, -14, -7, -6],
       ['C', -40, -33, -26, -19, -18, -11, -10],
       ['A', -44, -37, -30, -23, -22, -15, -14],
       ['G', -48, -41, -34, -27, -26, -19, -18],
       ['C', -52, -45, -38, -31, -30, -23, -22],
       ['C', -56, -49, -42, -35, -34, -27, -26],
       ['A', -60, -53, -46, -39, -38, -31, -30]], dtype=object)

In [25]:
def traceback(matrix, match=2, mismatch=-1, gap=-2):
    seq_X = [matrix[i, 0] for i in range(2, matrix.shape[0])]
    seq_Y = [matrix[0, j] for j in range(2, matrix.shape[1])]
    
    i, j = len(seq_X), len(seq_Y)
    total_score = matrix[i + 1, j + 1]
    
    align_X, align_Y = [], []
    
    while i > 0 or j > 0:
        current = matrix[i + 1, j + 1]
        
        # Диагональ
        if i > 0 and j > 0:
            char_X, char_Y = seq_X[i-1], seq_Y[j-1]
            expected = matrix[i, j] + (match if char_X == char_Y else mismatch)
            if abs(float(current) - float(expected)) < 0.001:
                align_X.append(char_X)
                align_Y.append(char_Y)
                i -= 1
                j -= 1
                continue
        
        # Вверх
        if i > 0 and abs(float(current) - (float(matrix[i, j + 1]) + gap)) < 0.001:
            align_X.append(seq_X[i-1])
            align_Y.append('-')
            i -= 1
            continue
        
        # Влево
        if j > 0 and abs(float(current) - (float(matrix[i + 1, j]) + gap)) < 0.001:
            align_X.append('-')
            align_Y.append(seq_Y[j-1])
            j -= 1
            continue
        
        # Если ничего не подошло - выбираем максимальный путь
        options = []
        if i > 0 and j > 0:
            options.append(('diag', matrix[i, j], i-1, j-1, seq_X[i-1], seq_Y[j-1]))
        if i > 0:
            options.append(('up', matrix[i, j + 1], i-1, j, seq_X[i-1], '-'))
        if j > 0:
            options.append(('left', matrix[i + 1, j], i, j-1, '-', seq_Y[j-1]))
        
        if options:
            best = max(options, key=lambda x: float(x[1]))
            align_X.append(best[4])
            align_Y.append(best[5])
            i, j = best[2], best[3]
        else:
            break
    
    align_X.reverse()
    align_Y.reverse()
    
    aligned_X = ''.join(map(str, align_X))
    aligned_Y = ''.join(map(str, align_Y))
    
    matches = ''.join(
        '|' if a == b else ' ' if a == '-' or b == '-' else '*'
        for a, b in zip(align_X, align_Y)
    )
    
    return aligned_X, aligned_Y, matches, total_score

In [26]:
traceback(needleman_wunsch(x, y, 3, -3, -4))

('ATGCAGCAGCAGCCA', 'ATATA---T------', '||**|   *      ', -30)

In [23]:
import numpy as np

x = 'ATGCAGCAGCAGCCA'
y = 'ATATAT'

def needleman_wunsch_affine(X, Y, match, mismatch, gap_open, gap_extend):
    X = list(X)
    Y = list(Y)

    n = len(X)
    m = len(Y)

    # main score matrix with headers (object, only int and str)
    matrix = np.empty((n + 2, m + 2), dtype=object)

    # put sequence letters in header row/column
    for i in range(n):
        matrix[i + 2, 0] = X[i]
    for j in range(m):
        matrix[0, j + 2] = Y[j]

    # separate DP matrices with dtype=int (no inf needed)
    S  = np.zeros((n + 1, m + 1), dtype=int)  # best overall
    Ix = np.zeros((n + 1, m + 1), dtype=int)  # gap in X (horizontal)
    Iy = np.zeros((n + 1, m + 1), dtype=int)  # gap in Y (vertical)

    # initialize first column/row with cumulative affine gaps
    for i in range(1, n + 1):
        S[i, 0]  = -(gap_open + (i - 1) * gap_extend)
        Iy[i, 0] = S[i, 0]
        Ix[i, 0] = 0  # impossible, but will be overwritten

    for j in range(1, m + 1):
        S[0, j]  = -(gap_open + (j - 1) * gap_extend)
        Ix[0, j] = S[0, j]
        Iy[0, j] = 0  # impossible, but will be overwritten

    # fill DP
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            score_sub = match if X[i - 1] == Y[j - 1] else mismatch
            S_diag = max(S[i - 1, j - 1], Ix[i - 1, j - 1], Iy[i - 1, j - 1]) + score_sub

            Ix[i, j] = max(
                S[i, j - 1] - (gap_open + gap_extend),
                Ix[i, j - 1] - gap_extend
            )

            Iy[i, j] = max(
                S[i - 1, j] - (gap_open + gap_extend),
                Iy[i - 1, j] - gap_extend
            )

            S[i, j] = max(S_diag, Ix[i, j], Iy[i, j])

    # copy scores into your visible matrix as plain ints
    for i in range(n + 1):
        for j in range(m + 1):
            matrix[i + 1, j + 1] = S[i, j]

    return matrix


In [24]:
needleman_wunsch_affine(x, y, 3, -3, -10, -1)

array([[None, None, 'A', 'T', 'A', 'T', 'A', 'T'],
       [None, np.int64(0), np.int64(10), np.int64(11), np.int64(12),
        np.int64(13), np.int64(14), np.int64(15)],
       ['A', np.int64(10), np.int64(21), np.int64(32), np.int64(43),
        np.int64(54), np.int64(65), np.int64(76)],
       ['T', np.int64(11), np.int64(32), np.int64(43), np.int64(54),
        np.int64(65), np.int64(76), np.int64(87)],
       ['G', np.int64(12), np.int64(43), np.int64(54), np.int64(65),
        np.int64(76), np.int64(87), np.int64(98)],
       ['C', np.int64(13), np.int64(54), np.int64(65), np.int64(76),
        np.int64(87), np.int64(98), np.int64(109)],
       ['A', np.int64(14), np.int64(65), np.int64(76), np.int64(87),
        np.int64(98), np.int64(109), np.int64(120)],
       ['G', np.int64(15), np.int64(76), np.int64(87), np.int64(98),
        np.int64(109), np.int64(120), np.int64(131)],
       ['C', np.int64(16), np.int64(87), np.int64(98), np.int64(109),
        np.int64(120), np.int64(1

In [27]:
def affine_traceback(matrix, gap_open, gap_extend):
    """
    Traceback for affine gap Needleman-Wunsch from bottom-right.
    Returns aligned sequences.
    """
    n = matrix.shape[0] - 2  # seq X length
    m = matrix.shape[1] - 2  # seq Y length
    
    align_X = []
    align_Y = []
    
    i, j = n + 1, m + 1  # bottom-right corner
    
    while i > 1 and j > 1:
        score = int(matrix[i, j])
        
        # get current chars
        char_X = matrix[i, 0]
        char_Y = matrix[0, j]
        
        # diagonal check (match/mismatch)
        diag_score = int(matrix[i - 1, j - 1]) + (1 if char_X == char_Y else -1)
        if score == diag_score:
            align_X.append(char_X)
            align_Y.append(char_Y)
            i -= 1
            j -= 1
            continue
        
        # check gap open in X (vertical move, delete from X)
        up_open = int(matrix[i - 1, j]) - (gap_open + gap_extend)
        if score == up_open:
            align_X.append(char_X)
            align_Y.append('-')
            i -= 1
            continue
        
        # check gap extend in X
        up_extend = int(matrix[i - 1, j]) - gap_extend
        if score == up_extend:
            align_X.append(char_X)
            align_Y.append('-')
            i -= 1
            continue
        
        # check gap open in Y (horizontal move, insert in X)
        left_open = int(matrix[i, j - 1]) - (gap_open + gap_extend)
        if score == left_open:
            align_X.append('-')
            align_Y.append(char_Y)
            j -= 1
            continue
        
        # check gap extend in Y
        left_extend = int(matrix[i, j - 1]) - gap_extend
        if score == left_extend:
            align_X.append('-')
            align_Y.append(char_Y)
            j -= 1
            continue
        
        # if no exact match, prefer diagonal (fallback)
        align_X.append(char_X)
        align_Y.append(char_Y)
        i -= 1
        j -= 1
    
    # add remaining prefix gaps
    while i > 1:
        align_X.append(matrix[i, 0])
        align_Y.append('-')
        i -= 1
    while j > 1:
        align_X.append('-')
        align_Y.append(matrix[0, j])
        j -= 1
    
    return ''.join(reversed(align_X)), ''.join(reversed(align_Y))


In [28]:
M = needleman_wunsch_affine(x, y, match=1, mismatch=-1, gap_open=10, gap_extend=1)
align1, align2 = affine_traceback(M, gap_open=10, gap_extend=1)
print(align1)
print(align2)

ATGCAGCAGCAGCCA
---------ATATAT


In [29]:
#скажу честно, я хз как эти трейсбеки самому писать, + какое-то говно получилось + я не шарю